---
## **0. 준비**
---

In [1]:
%%html
<style>
    body {
        /* change --vscode-font-family if it's different on your system */
        --vscode-font-family: "SeoulNamsanC";
    }
</style>

In [1]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import GoogleGenerativeAI

In [2]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [3]:
import config
config.set_environment()

---
## **로컬 모델 실행**
---

- 로컬 모델의 장점:
    - 데이터에 대한 완전한 통제권과 개인정보 보호 보장
    - API 비용과 사용량 제한에서 자유로움
    - 인터넷 연결 없이 사용 가능
    - 모델 매개변수와 미세 조정을 직접 제어

---
### **Ollama 시작하기**
- Ollama는 개발자 친화 도구로 강력한 오픈 소스 모델을 로컬 환경에서 실행할 수 있도록 해줌
- 다양한 모델을 다운로드하고 실행할 수 있는 직관적인 인터페이스 제공

In [4]:
# Ollama를 LCEL 패턴과 통합하는 방법 학습
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 선택한 모델로 Ollama 초기화
local_llm = ChatOllama(
    model = "qwen3.5:4B",
    base_url="http://172.30.128.1:11434",
    temperature=0,
)

# 로컬 모델을 사용하는 LECL 체인 생성
prompt = PromptTemplate.from_template("단순한 용어로 {concept}을 설명하라.")
local_chain = prompt | local_llm | StrOutputParser()

# 로컬 모델 체인 사용
result = local_chain.invoke({"concept": "양자 컴퓨팅"})
print(result)

양자 컴퓨팅을 가장 쉽게 설명하자면, **"일반 컴퓨터가 '0'과 '1' 중 하나를 선택하는 것이라면, 양자 컴퓨터는 '0'과 '1'을 동시에 처리할 수 있는 마법 같은 컴퓨터"**라고 할 수 있습니다.

핵심적인 차이를 3 가지 쉬운 비유로 정리해 드릴게요.

### 1. 스위치 vs. 돌린 동전
*   **일반 컴퓨터 (비트):** 전등 스위치처럼, '켜짐' (1) 이나 '꺼짐' (0) 중 하나만 있는 상태입니다. 계산할 때 이 두 가지 상태 중 하나를 선택해서 처리합니다.
*   **양자 컴퓨터 (큐비트):** 돌린 동전처럼, 앞면과 뒷면이 동시에 존재하는 상태입니다. 이를 **'중첩'**이라고 합니다. 그래서 한 번에 여러 가지 가능성을 모두 계산해 볼 수 있습니다.

### 2. 도서관 찾기 비유
*   **일반 컴퓨터:** 방대한 도서관에서 특정 책을 찾으려면, 책장 하나를 하나씩 확인해야 합니다. (서로 다른 책장을 동시에 보지 못함)
*   **양자 컴퓨터:** 모든 책장이 있는 도서관을 한 번에 훑어볼 수 있습니다. (중첩 덕분에 여러 가지 가능성을 동시에 탐색함)

### 3. 어떤 일을 할 때 유리할까?
양자 컴퓨터는 모든 일을 더 빠르거나 잘하는 것이 아닙니다. 대신, 일반 컴퓨터로는 풀기 너무 어려운 **'특정 복잡한 문제'**를 해결할 때 빛을 발합니다.
*   **예시:** 신약 개발 (분자 구조 분석), 암호 해독, 인공지능 학습 등

### 요약하자면
양자 컴퓨팅은 **"0 과 1 을 동시에 처리하는 능력"** 덕분에, 일반 컴퓨터로는 수천 년 걸릴 일을 몇 시간 만에 해결할 수 있는 잠재력을 가진 기술입니다. 하지만 아직 초기 단계이며, 일상적인 스마트폰이나 웹 브라우저와는 다른 용도로 쓰일 것입니다.


---
### **로컬에서 Hugging Face 모델 작업**
- 허깅페이스는 모델 실행을 위한 두 가지 주요 방식을 지원
    - 로컬 환경에서 직접 실행하는 HuggingFacePipeline
    - 클라우드 기반 Hugging Face Hub를 통해 실행하는 HuggingFaceEndpoint
- 해당 실습에서는 로컬 실행 방식에 집중

#### **모델 동작 제어**
- LLM은 생성 동작을 세밀하게 제어할 수 있는 여러 매개변수를 제공하며, 구체적인 구현은 공급자마다 다름

| 매개변수 | 설명 | 일반적인 범위 | 최상의 대상 |
|---|---|---|---|
| **Temperature** | 텍스트 생성 시 무작위성 제어 | 0.0 - 1.0 | 낮음(0.0 - 0.3) : 사실적 작업, Q&A<br>높음(0.7+) : 창의적 글쓰기, 브레인 스토밍 |
| **Top-k** | 가장 확률이 높은 k개의 토큰으로 선택 범위를 제한 | 1 - 100 | 낮은 값(1-10) : 집중된 출력<br>높은 값 : 다양한 완성도 |
| **Top-p** | 누적 확률이 임계값에 도달할 때까지 토큰 후보를 고려 | 0.0 - 1.0 | 낮은 값(0.5) : 더 집중된 출력<br>높은 값(0.9) : 더 다양한 아이디어와 창의적 응답 |
| **Max tokens** | 최대 응답 길이 제한 | 모델별 | 비용 관리 및 장황한 출력 방지 |
| **Presence/frequency penalties** | 이미 나타난 토큰에 패널티를 부여해 반복되는 것을 억제 | -2.0 - 2.0 | 반복이 바람직하지 않은 긴 글 생성시에 유용 |
| **Stop sequnences** | 모델에 생성 중지 시점을 알려줌 | 사용자 정의 문자열 | 정확한 생성 종료 지점 제어 |

- **Temperature + Top-k/Top-p** : 먼저 Top-k/Top-p가 토큰 분포를 필터링하고, 그 후 Temperature가 필터링된 집합 내에서 무작위성을 조절.
- **Penalties + Temperature** : 높은 Temperature와 낮은 Penalties를 함께 사용하면 창의적이지만 반복적인 텍스트가 생성될 수 있음.
- 엔터프라이즈 어플리케이션에서는 일관성과 정확성이 가장 중요하므로 이를 위해 낮은 Temperature(0.0-0.3)와 중간 범위의 Top_p(0.5-0.7)을 조합하는 것이 효과적임.

In [4]:
# 예시
from langchain_openai import OpenAI

# 사실 중심의 일관된 응답
fractual_llm = OpenAI(temperature=0.1, max_tokens=256)

# 창의적 브레인스토밍
creative_llm = OpenAI(temperature=0.8, top_p=0.95, max_tokens=512)

---
### **프롬프트와 템플릿**
**LangChain의 프롬프트 관리 시스템**은 크게 네 가지 기능을 갖추고 있음
- 동적 프롬프트 생성을 위한 **템플릿 시스템**
- 변경사항을 추적할 수 있는 **프롬프트 관리 및 버전 관리 기능**
- 모델 성능을 향상하기 위한 **퓨샷(few-shot) 예시 관리**
- 신뢰할 수 있는 결과를 도출하기 위한 **출력 파싱 및 검증**

> **LangChain의 프롬프트 템플릿은 정적 텍스트를 변수 치환이 가능한 동적 프롬프트로 변환**

> **또한, 템플릿의 주된 가치는 프롬프트를 LangChain 체인에서 공통 인터페이스로 다룰 수 있게 하는데 있음**

In [8]:
# 1. 동적 프롬프트 생성 함수를 직접 정의해서 활용
def generate_prompt(question, context=None):
    if context:
        return f"Context information: {context}\n\nAnswer this question concisely: {question}"
    return f"Answer this question concisely: {question}"

prompt_text = generate_prompt("프랑스의 수도는?", context="유럽의 국가 수도를 묻는 질문입니다.")
print(prompt_text)

# 모델에 직접 연결해야함
llm = ChatOpenAI(model='gpt-6-luna')
response = llm.invoke(prompt_text)
print(response.text)

Context information: 유럽의 국가 수도를 묻는 질문입니다.

Answer this question concisely: 프랑스의 수도는?
파리입니다.


In [9]:
# 2.LangChain의 PromptTemplate 활용
from langchain_core.prompts import PromptTemplate

# 한번 정의해 여러 곳에서 재사용
question_template = PromptTemplate(
    input_variables=["question", "context"],
    template="Context information: {context}\n\nAnswer this question concisely: {question}"
)

prompt_text = question_template.format(question="프랑스의 수도는?", context="유럽의 국가 수도를 묻는 질문입니다.")
print(prompt_text)

# 다른 langchain 구성요소와 연결 가능
chain = question_template | llm
response = chain.invoke({"question": "프랑스의 수도는?", "context": "유럽의 국가 수도를 묻는 질문입니다."})
print(response.text)

Context information: 유럽의 국가 수도를 묻는 질문입니다.

Answer this question concisely: 프랑스의 수도는?
프랑스의 수도는 파리입니다.


<hr style="border:0;border-top:1.5px dashed gray;">

#### **채팅 프롬프트 템플릿**
- 채팅 모델의 경우, 다양한 역할이 포함된 더욱 구조화된 프롬프트를 생성할 수 있음

In [7]:
template = ChatPromptTemplate.from_messages([
    ("system", "너는 한국어를 불어로 번역하는 번역기야."),
    ("human", "다음 말을 불어로 번역해줘: {text}")])

chat = ChatOpenAI(
    model="gpt-6-luna",
    use_responses_api=True,
    reasoning_effort="low", 
)

# 순차 적용 결과
formatted_messages = template.format_messages(text= "실존은 본질에 선행한다.")
response_1 = chat.invoke(formatted_messages)
print(response_1.text)

# chain으로 연결한 결과
chain = template | chat
response_2 = chain.invoke({"text": "실존은 본질에 선행한다."})
print(response_2.text)

L’existence précède l’essence.
L’existence précède l’essence.


---
### **LangChain 표현 언어(LECL)**
- LECL은 LangChain 기반 LLM 어플리케이션 개발 방식을 혁신적으로 진화시킨 기술
- 개발자는 원하는 최종 결과만 정의하면 실행 세부 사항은 LangChain이 자동으로 처리
- 파이프 연산자를 사용해 컴퍼넌트들을 연결하면 처리 흐름이 자동으로 구성 -> **Runnable 인터페에스 덕분**
- Runnable Interface :
    - 모든 컴퍼넌트가 표준화된 방식으로 입력을 처리하고 출력을 생성
    - 일관된 메서트 세트를 제공 :
        - `invoke()`: 단일 입력을 동기적으로 처리하고 출력을 반환
        - `stream()`: 출력이 생성되는 동안 실시간으로 스트림 형태로 전달
        - `batch()`: 다중 입력 병렬 처리
        - `ainvoke()`, `abatch()`, `astream()`: 위 메서드들의 비동기 버전

<hr style="border:0;border-top:1.5px dashed gray;">

#### **LECL을 이용한 간단한 워크플로**

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 컴포넌트 생성
prompt = PromptTemplate.from_template("다음에 대한 농담해줘 : {topic}")
llm = ChatOpenAI(model="gpt-6-luna", use_responses_api=True, reasoning_effort="low")
output_parser = StrOutputParser()

# LECL 체인 구성
chain = prompt | llm | output_parser

# 한 번의 호출로 워크플로 실행
result = chain.invoke({"topic": "프로그래밍"})
print(result)

프로그래머가 바다에 가면 가장 먼저 하는 일은?

파도를 **처리(handling)**하는 거야. 파도마다 예외가 생길 수 있으니까!


<hr style="border:0;border-top:1.5px dashed gray;">

#### **복잡한 체인의 예**

In [26]:
# 스토리 생성과 분석 사례 : 고급화된 패턴
"""
1. LLM 호출을 통해 초기 컨텐츠를 생성
2. 생성된 출력을 두번째 LLM 호출의 입력값으로 전달
3. 전체 체인 실행 과정에서 데이터의 지속성 유지 및 변환을 처리
"""

# 모델 초기화 
llm_1 = ChatOpenAI(
    model= "gpt-6-luna",
    reasoning_effort="none",                                                 #생략과 다름
    use_responses_api= True, temperature=0.8, top_p=0.95, max_tokens=512)
    # 추론 강도가 none이 아닐때는 temperature, top_p등을 제거해야함.

llm_2 = ChatOpenAI(
    model = "gpt-6-sol",
    reasoning_effort="medium",
    use_responses_api= True)
    
# 첫 번째 체인 : 이야기 생성
story_prompt = PromptTemplate.from_template("다음에 대한 스토리 작성 : {topic}")
story_chain = story_prompt | llm_1 | StrOutputParser()

# 두 번쩨 체인 : 이야기 분석
analysis_prompt = PromptTemplate.from_template(
    "이 이야기의 분위기를 분석해줘:\n{story}")
analysis_chain = analysis_prompt | llm_2 | StrOutputParser()

In [27]:
# 체인 결합
story_with_analysis = story_chain | analysis_chain

# 결합 체인 실행
story_analysis = story_with_analysis.invoke({"topic": "비 오는날"})

print("\nAnalysis:", story_analysis)


Analysis: 이 이야기는 **아쉬움에서 시작해 포근하고 희망적인 분위기로 바뀝니다.**

처음에는 소풍날 비가 내려 지우가 실망하고, 창문을 두드리는 빗소리도 그 마음을 더 쓸쓸하게 만듭니다. 하지만 민호가 찾아오면서 분위기가 달라집니다. 함께 우산을 쓰고 걷는 모습, 도서관의 따뜻한 코코아 냄새와 그림책 읽기 행사는 비 오는 날을 아늑하게 느끼게 합니다.

마지막의 “지우의 마음에는 작은 햇살이 비치고 있었다”는 표현은 **날씨는 그대로여도 지우의 마음은 밝아졌음**을 보여줍니다. 전체적으로 예상과 다른 하루에서도 즐거움을 발견하는, 따뜻하고 긍정적인 분위기의 이야기입니다.


> 위 방법은 작동하지만 한계가 있음 -> 분석 결과만 저장되고 **원본 스토리는 사라짐**

In [ ]:
# 1. RunnablePassthrough.assign으로 데이터 보존

from langchain_core.runnables import RunnablePassthrough
enhanced_chain = RunnablePassthrough.assign(
    story=story_chain                                                       #생성된 내용을 story 키로 추가
).assign(
    analysis=analysis_chain                                                 #스토리 분석 결과를 analysis 키로 추가
)
# 체인 실행
result = enhanced_chain.invoke({'topic':'비 오는날'})
print(result.keys())

dict_keys(['topic', 'story', 'analysis'])


In [ ]:
# 내용 확인
print(result['story'], "\n")
print("분석:\n", result['analysis'])

> 위 사례에서 딕셔너리를 직접 구성하여 출력 구조를 더 세밀하게 제어할 수 있음

In [ ]:
# 2. 딕셔너리 구성 방식 대안
from operator import itemgetter

manual_chain = (
    RunnablePassthrough() |                                                 #입력 그대로 전달
    {
        "story" : story_chain,                                              #스토리 결과 추가
        "topic" : itemgetter("topic")                                       #기존 topic 보존
    } |
    RunnablePassthrough().assign(analysis=analysis_chain)
)
result_manual = manual_chain.invoke({"topic":'비 오는날'})
print(result_manual.keys())

dict_keys(['story', 'topic', 'analysis'])


In [ ]:
# # 내용 확인
print(result_manual['story'], "\n")
print("분석:\n", result_manual['analysis'])

> 딕셔너리 변환을 사용해 LCEL 단축 표기법으로 위를 단순화할 수 있음

In [ ]:
# 3. 간단한 딕셔너리 구성
simple_dict_chain_corrected = story_chain | {
    "story": RunnablePassthrough(),                                      #스토리 출력을 'story'로 전달
    "analysis" : analysis_chain
}

# analysis_chain은 예상대로 {'story' :'...(생성된 이야기)'}을 수신
result_corrected = simple_dict_chain_corrected.invoke({'topic': '비 오는날'})
print(result_corrected.keys())

dict_keys(['story', 'analysis'])


In [ ]:
# 내용 확인
print(result_corrected['story'], "\n")
print("분석:\n", result_corrected['analysis'])

- **복수 LLM 호출** : 단일 프롬프트가 아닌, 여러 LLM 상호작용을 연결하여 파이프라인을 구성
- **데이터 변환** : `RunnablePassthrough`와 `itemgetter`를 활용하여 데이터를 효율적으로 관리하고 변환
- **딕셔너리 유지** : 단일 값을 넘어 전체 컨텍스트를 체인 전반에 걸쳐 유지
- **구조화된 출력** : 단순 문자열이 아닌 구조화된 딕셔너리 형태의 출력을 생성

> **LCEL**은 복잡한 워크플로를 우아하게 처리할 수 있지만, **상태 관리와 고급 분기 로직 같은 고급 기능은 한계가 있음 -> LangGraph 필요**